# photoD on DP2

Two commands and a check. The fit itself lives in `scripts/run_dp2.py`; there is nothing to reimplement here.

The prior maps, the dust curves and the locus come with the repository and are the ones the DP2 catalog
was built with, so a run here and a run at UW can be compared and nothing has to be fetched first.

What this does that a `merge_map` run does not:

- the prior maps are looked up by sightline instead of joined, because that join keeps one star partition per
  map pixel and drops the rest, which on DP2 is fifteen in sixteen and says nothing about it
- the locus is `LSSTlocus_10Gyr_DP2.txt`, whose u-g and main sequence are calibrated on DP2 stars
- `--floor` adds the error the locus itself carries, on top of the photometric systematic already in the input
- the A_r grid reaches past the extinction of the field instead of stopping at 2.5 mag
- the extinction is bounded by the column a 3D map measured rather than by SFD integrated to infinity
- every star comes back with a distance modulus and a flags column

## What to set

In [ ]:
photod   = "../photoD"                                   # the checkout, on fixed_fast
catalog  = "/mnt/beegfs/scratch/data/tmp_lsst_dp2/dp2_to_run/"
priors   = ""     # empty uses the maps that come with the repository
dust     = ""     # empty uses the curves that come with the repository
outPath  = "/mnt/beegfs/scratch/data/photoD_results"
outName  = "dp2-2"

arColumn = "Ar_SFD"   # the extinction column of the prepared catalog
arMax    = 8.0        # top of the A_r grid in magnitudes; the width itself costs nothing
workers  = 8          # they share whatever GPUs are there, two per card on four

## Run it

A partition is a file and fitting one has nothing to say to the next, so there is nothing to schedule: a pool
of processes reads, fits and writes one file per task and a worker is replaced every few hundred of them,
which is what keeps the memory of a survey-wide run flat.

If the prior maps are still a HATS catalog rather than a file, convert them once:

    python scripts/make_priors.py --from-catalog <the catalog> --out priors_dp2.npz

In [ ]:
%%time
import subprocess, sys

command = [
    sys.executable, f"{photod}/scripts/run_dp2.py",
    "--catalog", catalog,
    *(["--priors", priors] if priors else []),
    *(["--dust-curves", dust] if dust else []),
    "--out", outPath, "--name", outName,
    "--ar-column", arColumn,
    "--ar-max", str(arMax),
    "--workers", str(workers),
    "--overwrite",
]
print(" ".join(command))
run = subprocess.run(command, cwd=photod, text=True)
assert run.returncode == 0, f"the run failed with {run.returncode}"

## Did every star come out the other side

The check that catches a join quietly dropping partitions, or anything else losing rows. These two numbers
have to be equal.

In [ ]:
import glob
import pyarrow.parquet as pq
import lsdb
from hats.io.paths import pixel_catalog_file

base = lsdb.open_catalog(catalog).hc_structure.catalog_base_dir
starsIn = sum(pq.ParquetFile(str(pixel_catalog_file(base, p))).metadata.num_rows
              for p in lsdb.open_catalog(catalog).hc_structure.get_healpix_pixels())
starsOut = sum(pq.ParquetFile(f).metadata.num_rows
               for f in glob.glob(f"{outPath}/{outName}/dataset/**/*.parquet", recursive=True))
assert starsIn == starsOut, f"{starsIn:,} went in and {starsOut:,} came out: {starsIn - starsOut:,} lost"
print(f"{starsOut:,} stars in and out")

## What came out

`flags` is one bit per thing worth knowing: 1 the locus does not pass through the colours or the fit found
nothing, 2 the Mr posterior is lopsided so a giant and a dwarf solution both survived, 4 [Fe/H] is against the
end of the model grid, 8 A_r is against the top of its grid, 16 a colour had no measurement. Nothing is
dropped for being flagged. Bit 16 is normal rather than a fault, it is nearly all the u band, so the cut to
start from is `flags & 3 == 0`.

`DM` is the distance modulus, so the distance is `10 ** (DM / 5 + 1)` parsecs.

In [ ]:
import numpy as np

w = lsdb.open_catalog(f"{outPath}/{outName}")
print(w.npartitions, "partitions,", len(w.columns), "columns")
w.head(5)

In [ ]:
cols = ["flags", "chi2min", "DM_quantile_median", "Mr_quantile_median",
        "FeH_quantile_median", "Ar_quantile_median"]
files = sorted(glob.glob(f"{outPath}/{outName}/dataset/**/*.parquet", recursive=True))
d = {c: [] for c in cols}
for f in files[:: max(1, len(files) // 200)]:
    t = pq.read_table(f, columns=cols)
    for c in cols:
        d[c].append(t[c].to_numpy(zero_copy_only=False))
d = {c: np.concatenate(v) for c, v in d.items()}
flags = d["flags"].astype(int)
print(f"{len(flags):,} stars sampled, usable at flags & 3 == 0: {100 * np.mean(flags & 3 == 0):.2f} %")
for bit, label in ((1, "poor fit or no answer"), (2, "two branches"), (4, "FeH at the grid end"),
                   (8, "A_r at the grid top"), (16, "a colour missing")):
    print(f"  bit {bit:2d} {label:22s}: {100 * np.mean((flags & bit) > 0):6.2f} %")
for c in cols[2:]:
    v = d[c].astype(float)
    print(f"  {c:22s} 5/50/95 %: " + "  ".join(f"{x:7.3f}" for x in np.nanpercentile(v, [5, 50, 95])))